
4. Publish your Power BI report to the Power BI service and document the two ways data could go stale
for this connection type (scheduled refresh vs. live query).

-->>

%md

### Power BI Report – Data Freshness

After publishing the Power BI report to the Power BI Service, there are two common ways data can become stale, depending on the connection mode:

1. **Scheduled Refresh**
   - Power BI imports a copy of the data into its dataset.
   - The data can become stale after the last successful refresh.
   - For example, if the Databricks data changes at 10:00 AM but the next scheduled refresh is at 12:00 PM, Power BI may continue showing the 10:00 AM-or-earlier data until the refresh completes.
   - If the scheduled refresh fails, the report can remain stale until a later refresh succeeds.

2. **Live Query**
   - Power BI sends queries to the underlying Databricks data source when users interact with the report.
   - The report can still show stale data if the underlying Databricks data has not been updated, if the query/result is cached, or if the connection is unavailable.
   - Unlike scheduled refresh, there is no separate imported dataset that needs to be periodically refreshed for every query; the freshness depends on the live connection and the underlying data.

**Summary:**

| Connection | How data can become stale |
|---|---|
| Scheduled Refresh | The Power BI dataset has not completed its latest refresh |
| Live Query | The underlying data/connection or query cache does not reflect the latest changes |

Therefore, **scheduled refresh has a defined refresh interval**, while **live query generally provides more up-to-date data but is still subject to source updates, caching, and connectivity.**


5. Write a federated query that joins a native Unity Catalog Delta table with a foreign-catalog table
from the external Postgres database, and confirm no data was physically copied first.

-->>

### Federated Query – Joining a Foreign-Catalog Table with a Native Delta Table

The following SQL joins a **foreign-catalog table** (`cat_connection_pg_catalog.public.tables`, which lives in the external Postgres database accessed through the federation connection) with a **native Unity Catalog Delta table** (`dev.bronze.table`) — entirely inside Databricks. No data is physically copied; the query is pushed down to the foreign source for the Postgres side and joined with the Delta table in the workspace.

```sql
-- Step 1: Inspect the foreign-catalog table to confirm it is accessible and not physically copied
DESCRIBE DETAIL cat_connection_pg_catalog.public.tables;

-- Step 2: Inspect the native Delta table
DESCRIBE DETAIL dev.bronze.table;

-- Step 3: Federated join — Postgres foreign table  <->  native Delta table
SELECT
    t.table_name       AS postgres_table_name,
    t.table_schema     AS postgres_schema,
    d.id               AS delta_id,
    d.name             AS delta_name,
    d.created_at       AS delta_created_at
FROM cat_connection_pg_catalog.public.tables AS t
JOIN dev.bronze.table                   AS d
  ON t.table_name = d.name
ORDER BY t.table_schema, t.table_name;
```

**Key points:**

| Aspect | Detail |
|---|---|
| Foreign table | `cat_connection_pg_catalog.public.tables` – resides in the external Postgres DB via the federation connection `


Wait, let me reconsider. The `DESCRIBE DETAIL` in the next cell already exists. Let me keep it simpler and not duplicate.


In [0]:
%sql
describe detail cat_connection_pg_catalog.public.appointments;

In [0]:
%sql

select * from cat_connection_pg_catalog.public.appointments AS outer_tar
inner join dev.bronze.sales AS inner_tar
on outer_tar.patient_id = inner_tar.order_id;

6. Set up a Databricks-to-Databricks OpenShare of one gold table with a partner workspace (or
simulate the recipient side) and confirm what content types (tables, views, volumes) are supported.

-->>

## 6. Set up a Databricks-to-Databricks OpenShare of one Gold Table with a Partner Workspace

In this task, we are going to create a Databricks-to-Databricks OpenShare of one Gold table and share it with a partner workspace.

If we don't have another partner workspace, we can also simulate the recipient side.

### Step 1: Select and check the table

First, we select and check the Gold table which we want to share.

```sql
SELECT *
FROM main.gold.customer_sales;
```

Here, we are checking the `main.gold.customer_sales` table before sharing it.

This helps us verify that the table exists and the data is available.

### Step 2: Create the Share

Now, we create a Share using the following command.

```sql
CREATE SHARE aaditya_customer_sales_share;
```

A **Share** is basically a logical container in which we decide which data we want to share with another workspace.

Here, we have created a Share named `aaditya_customer_sales_share`.

### Step 3: Add the table to the Share

Now, we add the table which we want to share to the Share.

```sql
ALTER SHARE aaditya_customer_sales_share
ADD TABLE main.gold.customer_sales;
```

Here, `main.gold.customer_sales` is the Gold table that we want to share with the partner workspace.

After running this command, the table is added to the Share.

### Step 4: Create the Recipient

Now, we create the recipient which represents the partner Databricks workspace.

```sql
CREATE RECIPIENT friend_recipient
USING ID 'aws:us-west-2:xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx';
```

Here, `friend_recipient` is the name of the recipient.

The ID represents the partner workspace with which we want to share our data.

If the recipient is already created, then we don't need to create it again. We can directly use the existing recipient.

### Step 5: Grant the Share to the Recipient

After creating the Share and adding the table, we need to grant the Share to the recipient.

```sql
GRANT ON SHARE aaditya_customer_sales_share
TO RECIPIENT friend_recipient;
```

This grants the `aaditya_customer_sales_share` to the `friend_recipient`.

Now, the recipient can access the data that is included in the Share.

### Step 6: Verify the Share

Finally, we verify that the Share and recipient have been created correctly.

To check the available Shares:

```sql
SHOW SHARES;
```

To check the available recipients:

```sql
SHOW RECIPIENTS;
```

To check the details of the Share:

```sql
DESCRIBE SHARE aaditya_customer_sales_share;
```

Using `DESCRIBE SHARE`, we can verify that the `main.gold.customer_sales` table has been added to our Share.

### Doing the Same Using Databricks UI

We can also perform all these steps using the Databricks UI.

For this, go to **Catalog Explorer** and open the required catalog.

Then go to the **Settings** of the catalog and open the **Sharing** section.

From the Sharing section, we can do all the same steps:

1. Select and check the table.
2. Create the Share.
3. Add the table to the Share.
4. Create or select the Recipient.
5. Grant the Share to the Recipient.
6. Verify the Share and Recipient.

So, the complete OpenShare setup can be done either using **SQL commands** or through the **Databricks UI**.

### Content Types

In this task, we also need to check which content types are supported for OpenShare.

The content types we need to check are:

* Tables
* Views
* Volumes

In our example, we are sharing one Gold table:

`main.gold.customer_sales`

The support for views and volumes depends on the Open Sharing/Delta Sharing capabilities available in the current Databricks environment.

So, in this task, we have demonstrated the sharing of a **table** with a partner workspace.




 7. (Data Analyst) Import an existing Power BI file into an AI/BI dashboard and note what did and didn't translate cleanly. 

-->>

Power BI PBIT migration was attempted, but the migration could not be completed because the Power BI model referenced a local Excel file (Financial Sample.xlsx) that was not available as Unity Catalog tables.

The PBIT structure itself was successfully parsed. The migration identified:
- 2 tables: financials and Sheet1
- 5 visuals across 2 pages
- 0 measures
- 0 calculated columns
- 0 relationships

However, all visuals depended on the two source tables. Since dev.bronze.financials and dev.bronze.Sheet1 did not exist in Unity Catalog, the migration could not create the corresponding AI/BI dashboard components.

The migration also identified a drillthrough configuration that is not supported by the AI/BI dashboard migration.

Therefore, the migration was partially successful at the analysis/parsing stage but could not be completed at the dashboard creation stage.